# 21 — Exploration versus exploitation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/021-exploration-vs-exploitation.ipynb)

Every learner faces the same trade: pull the arm that has paid so far, or pull an untried one and learn something. In this notebook you run epsilon-greedy on a three-armed bandit, measure the cost of guessing with regret, and meet the agent that gets stuck because it never guesses at all.

## 1. The numbers: three arms, three means

Each arm pays a noisy reward drawn around its own mean, and those means are hidden from the agent. Arm 2 is the best at 0.8 per pull; the other two are worse, and one sample from each arm is far too noisy to tell them apart.

In [ ]:
import numpy as np                              # the array library, preinstalled on Colab
rng = np.random.default_rng(0)                  # a seeded generator: same numbers every run
means = np.array([0.3, 0.5, 0.8])               # the hidden average payoff of each arm
optimal = means.max()                           # the best possible average, 0.8
print("true means:", means)                     # what the agent never sees
print("best arm:", int(means.argmax()))         # arm 2
print("one sample per arm:", np.round(rng.normal(means, 0.4), 2))   # noise makes the order unclear
print("best possible reward per pull:", optimal)  # the yardstick for regret

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `np.random.default_rng(0)` — a seeded generator, so your numbers match mine.
- `means = ...` — the hidden average payoff of each of the three arms.
- `optimal = means.max()` — the best average payoff available.
- `rng.normal(means, 0.4)` — one noisy draw per arm, spread of 0.4 around the mean.
- `print(...)` — the hidden truth next to a single round of evidence.

## 2. Step by step: one epsilon-greedy step at a time

The rule is two lines: with probability epsilon pick an arm at random, otherwise pick the arm with the best average so far. Each pull updates only that arm's average, so the estimates start noisy and settle down as the counts grow.

In [ ]:
eps = 0.1                                       # 10% of pulls are random
Q = np.zeros(3)                                 # estimated value of each arm, all zero at first
counts = np.zeros(3)                            # how many times each arm has been pulled
for t in range(30):                             # thirty pulls is enough to watch the rule work
    if rng.random() < eps:                      # roll the dice: explore?
        a = int(rng.integers(3))                # a uniformly random arm
        how = "explore"                        # remember why, for the print below
    else:                                       # otherwise exploit
        a = int(np.argmax(Q))                   # the arm with the highest estimate
        how = "exploit"                        # remember why, for the print below
    reward = rng.normal(means[a], 0.4)          # the noisy payoff for this pull
    counts[a] += 1                              # one more sample for this arm
    Q[a] += (reward - Q[a]) / counts[a]         # running mean: the new estimate for this arm
    if t < 10:                                  # print only the first few, to keep it readable
        print(f"step {t}: {how} arm {a} reward {reward:+.2f} Q {np.round(Q, 2)}")   # trace
print("Q after 30 pulls:", np.round(Q, 2))      # crude estimates, still moving
print("pulls per arm:", counts.astype(int))     # where the pulls went

**Line by line**

- `eps = 0.1` — the exploration rate: one pull in ten is random.
- `Q = np.zeros(3)` — the estimate of each arm's mean, starting with no information.
- `counts = np.zeros(3)` — needed to turn the sum of rewards into an average.
- `if rng.random() < eps` — the coin flip that decides explore versus exploit.
- `a = int(np.argmax(Q))` — the greedy choice: the arm with the best estimate.
- `reward = rng.normal(means[a], 0.4)` — the environment answers with noise.
- `Q[a] += (reward - Q[a]) / counts[a]` — incremental average, no need to store every reward.
- `print(...)` — the trace of choices and estimates for the first ten pulls.

## 3. Plot: regret is the bill for exploring

Regret is what the best arm would have paid minus what you actually got, added up over time. A learner that guesses at random pays about 0.27 per pull forever; a learner that converges on arm 2 has a regret curve that bends flat.

In [ ]:
import matplotlib.pyplot as plt                # plotting library, preinstalled on Colab

rng = np.random.default_rng(7)                  # fresh seed for the long run
eps = 0.1                                       # keep the same exploration rate
Q = np.zeros(3)                                 # estimates, reset
counts = np.zeros(3)                            # counts, reset
regret = []                                     # cumulative regret after each pull
total = 0.0                                     # running regret
for t in range(500):                            # five hundred pulls
    if rng.random() < eps:                      # explore with probability eps
        a = int(rng.integers(3))                # random arm
    else:                                       # exploit otherwise
        a = int(np.argmax(Q))                   # best known arm
    reward = rng.normal(means[a], 0.4)          # what this pull actually paid
    counts[a] += 1                              # update this arm's count
    Q[a] += (reward - Q[a]) / counts[a]         # update this arm's average
    total += optimal - reward                   # regret of this pull: the gap to the best arm
    regret.append(total)                        # store the running total

plt.figure(figsize=(6, 3))                      # a wide, short figure
plt.plot(regret, color="tab:blue", label="epsilon = 0.1")            # what we measured
plt.plot([0, 499], [0, 499 * (optimal - means.mean())], color="grey",  # the random-guessing baseline
         linestyle="--", label="pure guessing")                      # the random baseline
plt.xlabel("pull number")                       # x-axis label
plt.ylabel("cumulative regret")                 # y-axis label
plt.legend(loc="upper left")                    # name the two lines
plt.title("the curve bends flat: exploration stops costing once the arm is known")   # takeaway
plt.show()                                      # render the figure

print("final Q:", np.round(Q, 2))               # should be closest to arm 2
print("pulls per arm:", counts.astype(int))     # most of them on arm 2
print("total regret:", round(regret[-1], 1))    # far below the guessing line

**Line by line**

- `import matplotlib.pyplot as plt` — the plotting library, aliased to `plt`.
- `rng = np.random.default_rng(7)` — a new seed for this longer experiment.
- `regret = []` — one cumulative number per pull.
- `total += optimal - reward` — this pull's shortfall against the best arm.
- `plt.plot(regret, ...)` — the measured curve.
- `499 * (optimal - means.mean())` — the slope of pure guessing: the average gap per pull.
- `print(...)` — final estimates, pull counts, and the total bill.

## 4. Interactive: drag epsilon

Small epsilon is cheap but slow to find the good arm; large epsilon finds it fast and then keeps wasting pulls. Drag the slider and watch the height of the whole regret curve move, with the final bill printed in the title.

In [ ]:
from ipywidgets import interact               # sliders that rerun a function

def show(eps=0.1):                              # epsilon comes from the slider
    r = np.random.default_rng(7)                # same seed every drag, so only eps changes
    q = np.zeros(3)                             # estimates
    c = np.zeros(3)                             # counts
    curve = []                                  # regret after each pull
    tot = 0.0                                   # running regret
    for t in range(400):                        # 400 pulls keeps each drag instant
        if r.random() < eps:                    # explore
            a = int(r.integers(3))              # random arm
        else:                                   # exploit
            a = int(np.argmax(q))               # best known arm
        rew = r.normal(means[a], 0.4)           # noisy payoff
        c[a] += 1                               # count this pull
        q[a] += (rew - q[a]) / c[a]             # running average for this arm
        tot += optimal - rew                    # accumulate regret
        curve.append(tot)                       # store it
    plt.figure(figsize=(6, 3))                  # fresh figure for each drag
    plt.plot(curve, color="tab:blue")           # regret curve
    plt.plot([0, 399], [0, 399 * (optimal - means.mean())], color="grey", linestyle="--")   # guessing
    plt.ylim(0, 120)                            # fixed axis, so curves compare fairly
    plt.xlabel("pull number")                   # x-axis label
    plt.ylabel("cumulative regret")             # y-axis label
    plt.title(f"eps={eps:.2f} -> regret {tot:.0f}, favourite arm {int(np.argmax(q))}")   # readout
    plt.show()                                  # draw it

interact(show, eps=(0.0, 0.6, 0.05))            # the slider: range and step

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(eps=0.1)` — the function that redraws one frame.
- `np.random.default_rng(7)` — the same seed each drag, so the comparison is honest.
- `if r.random() < eps` — the explore branch, now at the slider's rate.
- `tot += optimal - rew` — regret accumulated across the 400 pulls.
- `plt.ylim(0, 120)` — a fixed window, otherwise every curve looks identical.
- `plt.title(f"...")` — the readout: total regret and the arm it settled on.
- `interact(show, eps=(0.0, 0.6, 0.05))` — attach the slider and its step size.

## 5. When it breaks: epsilon = 0 gets stuck

With epsilon at zero the agent only ever pulls its current favourite. If the first sample from arm 0 happened to be a lucky 1.0, that arm looks unbeatable, the other two are never tried, and their estimates stay at the optimistic zero they started with. The agent is confident, wrong, and unable to find out.

In [ ]:
rng = np.random.default_rng(11)                 # seed for the stuck run
Q = np.zeros(3)                                 # no information about any arm
Q[0] = 1.0                                      # except arm 0's first sample, a lucky 1.0
counts = np.zeros(3)                            # counts for the running averages
counts[0] = 1                                   # that lucky sample counts as one pull
stuck = []                                      # the greedy agent's regret curve
total = 0.0                                     # running regret
for t in range(400):                            # four hundred pulls with epsilon = 0
    a = int(np.argmax(Q))                       # always the favourite: no exploration at all
    reward = rng.normal(means[a], 0.4)          # the payoff, noisy as always
    counts[a] += 1                              # one more pull for arm 0
    Q[a] += (reward - Q[a]) / counts[a]         # its average drifts down towards 0.3
    total += optimal - reward                   # but it stays above the untried zeros
    stuck.append(total)                         # store the running total

rng = np.random.default_rng(11)                 # same seed, now with exploration
Qe = np.zeros(3)                                # fresh estimates
ce = np.zeros(3)                                # fresh counts
smart = []                                      # the exploring agent's regret curve
total = 0.0                                     # running regret
for t in range(400):                            # four hundred pulls with epsilon = 0.1
    if rng.random() < 0.1:                      # one pull in ten is random
        a = int(rng.integers(3))                # so the untried arms do get tried
    else:                                       # otherwise
        a = int(np.argmax(Qe))                  # same greedy rule as the stuck agent
    reward = rng.normal(means[a], 0.4)          # payoff
    ce[a] += 1                                  # count
    Qe[a] += (reward - Qe[a]) / ce[a]           # running average
    total += optimal - reward                   # regret
    smart.append(total)                         # store it

print("stuck agent's Q:", np.round(Q, 2))       # arm 0 near 0.3, arms 1 and 2 still 0.0
print("stuck agent's pulls:", counts.astype(int))   # 401, 0, 0: arms 1 and 2 never tried
print("regret after 400 pulls: stuck", round(stuck[-1], 1), "vs exploring", round(smart[-1], 1))   # about 193 vs 16

plt.figure(figsize=(6, 3))                      # compare the two curves
plt.plot(stuck, color="tab:red", label="epsilon = 0 (stuck)")        # the failure
plt.plot(smart, color="tab:green", label="epsilon = 0.1")            # the fix
plt.xlabel("pull number")                       # x-axis label
plt.ylabel("cumulative regret")                 # y-axis label
plt.legend(loc="upper left")                    # name the lines
plt.title("no exploration means a straight line of regret forever")   # the takeaway
plt.show()                                      # render

**Line by line**

- `Q[0] = 1.0` — a single lucky sample, which is all a greedy agent needs.
- `counts[0] = 1` — the lucky sample counts as the first pull.
- `a = int(np.argmax(Q))` — with epsilon at zero this is the only rule in the loop.
- `Q[a] += ...` — the average drifts down to 0.3 but never below the untried zeros.
- `if rng.random() < 0.1` — the exploring agent, identical except for this branch.
- `print(...)` — the stuck agent's estimates and its 401, 0, 0 pull counts.
- `plt.plot(stuck, ...)` — the straight line of regret against the flattening one.

## Your turn

Raise the slider's epsilon to 0.6 and predict the total regret before you look. Then work out the epsilon that would make the exploring agent in section 5 as bad as the stuck one — what part of every pull are you throwing away?